In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



<Accordion>
<AccordionItem title="גרסאות חבילות">

הקוד בדף זה פותח תוך שימוש בדרישות הבאות.
אנחנו ממליצים להשתמש בגרסאות אלה או בגרסאות חדשות יותר.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>

אפשר להשתמש באפשרויות כדי להתאים אישית את הפרימיטיב Estimator. בעוד שהממשק של שיטת `run()` של הפרימיטיבים משותף לכל המימושים, האפשרויות שלהם אינן כך. עיין במקורות ה-API לקבלת מידע על אפשרויות [`qiskit.primitives.BaseEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV2) ו-[`qiskit_aer.BaseEstimatorV2`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.primitives.EstimatorV2.html).

<span id="pass-options"></span>

## הגדרת אפשרויות Estimator

אפשר להגדיר אפשרויות בעת אתחול Estimator, אחרי אתחולו, או לעדכן את האפשרויות לאחר שה-Estimator אותחל. להוראות שימוש בטכניקות אלה, ראה את הנושא [מבוא לאפשרויות](/guides/runtime-options-overview#options-precedence).

בנוסף, אפשר להגדיר את ערך `precision` בשיטת `run()`, כפי שמתואר בסעיף הבא.

<span id="run-method"></span>

### שיטת Run()

הערכים היחידים שאפשר להעביר ל-`run()` הם אלה שמוגדרים בממשק. כלומר, `precision` עבור Estimator. זה מחליף כל ערך שנקבע עבור `default_precision` בריצה הנוכחית.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EstimatorOptions

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

options = EstimatorOptions(
    resilience_level=2,
    resilience={"zne_mitigation": True, "zne": {"noise_factors": [1, 3, 5]}},
)

# or...
options = EstimatorOptions()
options.resilience_level = 2
options.resilience.zne_mitigation = True
options.resilience.zne.noise_factors = [1, 3, 5]

estimator = Estimator(mode=backend, options=options)

#### Dictionary

You can specify options as a dictionary when initializing Estimator.

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during initialization
estimator = Estimator(
    backend,
    options={
        "resilience_level": 2,
        "resilience": {
            "zne_mitigation": True,
            "zne": {"noise_factors": [1, 3, 5]},
        },
    },
)

### מקרה מיוחד: דיוק
שיטת `EstimatorV2.run` מקבלת שתי ארגומנטים: רשימה של PUBs, שכל אחד מהם יכול לציין ערך דיוק ספציפי ל-PUB, וארגומנט keyword של דיוק. ערכי הדיוק האלה הם חלק מממשק הביצוע של Estimator, ועצמאיים מאפשרויות ה-Runtime Estimator. הם גוברים על כל ערך שצוין כאפשרויות כדי לציית לאבסטרקציה של Estimator.

אם `precision` לא צוין על ידי אף PUB ולא בארגומנט keyword של הריצה (או שכולם `None`), אזי ישמש ערך הדיוק מהאפשרויות, בעיקר `default_precision`.

שים לב שאפשרויות Estimator מכילות הן `default_shots` והן `default_precision`. עם זאת, מכיוון שgate-twirling מופעל כברירת מחדל, מכפלת `num_randomizations` ו-`shots_per_randomization` גוברת על שתי אפשרויות אלה.

ספציפית, עבור כל Estimator PUB:

1. אם ה-PUB מציין דיוק, השתמש בערך הזה.
2. אם ארגומנט keyword של דיוק צוין ב-`run`, השתמש בערך הזה.
3. אם `twirling` מופעל (True כברירת מחדל), אזי משמש מכפלת `num_randomizations` ו-`shots_per_randomization`, כפי שצוינו ב-[אפשרויות `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options).
4. אם `estimator.options.default_shots` צוין, השתמש בערך הזה לשליטה בכמות הנתונים.
5. אם `estimator.options.default_precision` צוין, השתמש בערך הזה.

לדוגמה, אם הדיוק צוין בכל ארבעת המקומות, ישמש זה בעל העדיפות הגבוהה ביותר (דיוק שצוין ב-PUB).

> **Note:** הדיוק משתנה בצורה הפוכה לשימוש. כלומר, ככל שהדיוק נמוך יותר, כך לוקח יותר זמן QPU להריץ.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

# Setting options after initialization
# This uses auto-complete.
estimator.options.default_precision = 0.01
# This does bulk update.
estimator.options.update(
    default_precision=0.02, resilience={"zne_mitigation": True}
)

<span id="run-method"></span>
### Run() method

The only values you can pass to `run()` are those defined in the interface.  That is, `precision` for Estimator. This overwrites any value set for `default_precision` for the current run.

In [16]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

observable = SparsePauliOp("Z" * 3)

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)
isa_observable1 = observable.apply_layout(transpiled1.layout)
isa_observable2 = observable.apply_layout(transpiled2.layout)

estimator = Estimator(mode=backend)
# Default precision to use if not specified in run()
estimator.options.default_precision = 0.01
# Run two circuits, requiring a precision of .02 for both.
estimator.run(
    [(transpiled1, isa_observable1), (transpiled2, isa_observable2)],
    precision=0.02,
)

<RuntimeJobV2('d7amh42k86tc73a1sa20', 'estimator')>

<span id="no-error-mitigation"></span>
## כיבוי כל הפחתת השגיאות וסיפוי השגיאות
אפשר לכבות את כל הפחתת הסיפוי של שגיאות אם אתה, לדוגמה, עורך מחקר על טכניקות ההפחתה שלך עצמך. לשם כך, הגדר `resilience_level = 0`.

דוגמה:

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

observable = SparsePauliOp("Z" * 3)

circuit = random_iqp(3)
circuit.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

isa_circuit = pass_manager.run(circuit)
isa_observable = observable.apply_layout(isa_circuit.layout)

# Setting precision during primitive initialization
estimator = Estimator(mode=backend, options={"default_precision": 0.05})

# Run with precision=0.02, overwriting the default.
estimator.run(
    [(isa_circuit, isa_observable1)],
    precision=0.02,
)

<RuntimeJobV2('d8286b00bvlc73d1vn50', 'estimator')>

<span id="options-table"></span>
## האפשרויות הזמינות
הטבלה הבאה מתעדת אפשרויות מהגרסה האחרונה של `qiskit-ibm-runtime`. כדי לראות גרסאות אפשרויות ישנות יותר, בקר ב-[מקור ה-API של `qiskit-ibm-runtime`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime) ובחר גרסה קודמת.

<Accordion>
<AccordionItem title="`default_shots`">

המספר הכולל של shots לשימוש לכל מעגל לכל תצורה.

**אפשרויות**: מספר שלם >= 0

**ברירת מחדל**: None

[תיעוד API של `default_shots`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#default_shots)
  </AccordionItem>

<AccordionItem title="`default_precision`">

הדיוק הברירת-מחדל לשימוש עבור כל PUB או קריאת `run()` שאינה מציינת אחד.

**אפשרויות**: Float > 0

**ברירת מחדל**: 0.015625 (1 / sqrt(4096))

[תיעוד API של `default_precision`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#default_precision)
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling`">

שליטה בהגדרות הפחתת שגיאות של dynamical decoupling.

[תיעוד API של `dynamical_decoupling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#dynamical_decoupling)
<Accordion>
<AccordionItem title="`dynamical_decoupling.enable`">

**אפשרויות**: `True`, `False`

**ברירת מחדל**: `False`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.extra_slack_distribution`">

**אפשרויות**: `middle`, `edges`

**ברירת מחדל**: `middle`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.scheduling_method`">

אפשרויות: `asap`, `alap`
ברירת מחדל: `alap`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.sequence_type`">

אפשרויות: `XX`, `XpXm`, `XY4`
ברירת מחדל: `XX`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.skip_reset_qubits`">

אפשרויות: `True`, `False`
ברירת מחדל: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`environment`">

[תיעוד API של `environment`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#environment)
<Accordion>
<AccordionItem title="`environment.callback`">

פונקציה קריאה המקבלת את `Job ID` ואת `Job result`.

**אפשרויות**: None

**ברירת מחדל**: None
  </AccordionItem>

<AccordionItem title="`environment.job_tags`">

רשימת תגיות.

**אפשרויות**: None

**ברירת מחדל**: None
  </AccordionItem>

<AccordionItem title="`environment.log_level`">

**אפשרויות**: DEBUG, INFO, WARNING, ERROR, CRITICAL

**ברירת מחדל**: WARNING
  </AccordionItem>

<AccordionItem title="`environment.private`">

**אפשרויות**: `True`, `False`

**ברירת מחדל**: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`execution`">

[תיעוד API של `execution`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#execution)
<Accordion>
<AccordionItem title="`execution.init_qubits`">
האם לאפס את הקיוביטים למצב היסוד לכל shot.

**אפשרויות**: `True`, `False`

**ברירת מחדל**: `True`
  </AccordionItem>

<AccordionItem title="`execution.rep_delay`">
ההשהיה בין מדידה למעגל קוונטי הבא.

**אפשרויות**: ערך בטווח שסופק על ידי `backend.rep_delay_range`

**ברירת מחדל**: נקבע על ידי `backend.default_rep_delay`
  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`max_execution_time`">
מגביל כמה זמן המשימה יכולה לרוץ, בשניות. ראה את המדריך [זמן ביצוע מקסימלי](/guides/max-execution-time) לפרטים.

**אפשרויות**: מספר שלם של שניות בטווח [1, 10800]

**ברירת מחדל**: 10800 (3 שעות)
  </AccordionItem>

<AccordionItem title="`resilience`">
אפשרויות מתקדמות של חוסן לכוונון עדין של אסטרטגיית החוסן.

[תיעוד API של `resilience`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience)
<Accordion>
<AccordionItem title="`resilience.layer_noise_learning`">

אפשרויות ללמידת רעש שכבה.

[תיעוד API של `resilience.layer_noise_learning`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-layer-noise-learning-options)
  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.layer_pair_depths`">

**אפשרויות**: list[int] של 2-10 ערכים בטווח [0, 200]

**ברירת מחדל**: `(0, 1, 2, 4, 16, 32)`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.max_layers_to_learn`">

**אפשרויות**: None, מספר שלם >= 1

**ברירת מחדל**: `4`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.num_randomizations`">

**אפשרויות**: מספר שלם >= 1

**ברירת מחדל**: `32`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.shots_per_randomization`">

**אפשרויות**: מספר שלם >= 1

**ברירת מחדל**: `128`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_model`">

**אפשרויות**: `NoiseLearnerResult`, `Sequence[LayerError]`

**ברירת מחדל**: None

  </AccordionItem>

<AccordionItem title="`resilience.measure_mitigation`">

**אפשרויות**: `True`, `False`

**ברירת מחדל**: `True`

  </AccordionItem>

<AccordionItem title="`resilience.measure_noise_learning`">

אפשרויות ללמידת רעש מדידה.

[תיעוד API של `resilience.measure_noise_learning`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-measure-noise-learning-options)
  </AccordionItem>

<AccordionItem title="`resilience.measure_noise_learning.num_randomizations`">

**אפשרויות**: מספר שלם >= 1

**ברירת מחדל**: `32`

  </AccordionItem>

<AccordionItem title="`resilience.measure_noise_learning.shots_per_randomization`">

**אפשרויות**: מספר שלם, `auto`

**ברירת מחדל**: `auto`

  </AccordionItem>

<AccordionItem title="`resilience.pec_mitigation`">

**אפשרויות**: `True`, `False`

**ברירת מחדל**: `False`

  </AccordionItem>

<AccordionItem title="`resilience.pec`">

אפשרויות הפחתת שגיאות הסתברותיות.

[תיעוד API של `resilience.pec`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-pec-options)
  </AccordionItem>

<AccordionItem title="`resilience.pec.max_overhead`">

**אפשרויות**: `None`, מספר שלם >= 1

**ברירת מחדל**: `100`

  </AccordionItem>

<AccordionItem title="`resilience.pec.noise_gain`">

**אפשרויות**: `auto`, float בטווח [0, 1]

**ברירת מחדל**: `auto`

  </AccordionItem>

<AccordionItem title="`resilience.zne_mitigation`">

**אפשרויות**: `True`, `False`

**ברירת מחדל**: `False`

  </AccordionItem>

<AccordionItem title="`resilience.zne`">

[תיעוד API של `resilience.zne`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)
  </AccordionItem>

<AccordionItem title="`resilience.zne.amplifier`">

**אפשרויות**: `gate_folding`, `gate_folding_front`, `gate_folding_back`, `pea`

**ברירת מחדל**: `gate_folding`

  </AccordionItem>

<AccordionItem title="`resilience.zne.extrapolated_noise_factors`">

**אפשרויות**: רשימת floats

**ברירת מחדל**: `[0, *noise_factors]`

  </AccordionItem>

<AccordionItem title="`resilience.zne.extrapolator`">

**אפשרויות**: אחד או יותר מ: `exponential`, `linear`, `double_exponential`, `polynomial_degree_(1 <= k <= 7)`, `fallback`

**ברירת מחדל**: `(exponential, linear)`

  </AccordionItem>

<AccordionItem title="`resilience.zne.noise_factors`">

**אפשרויות**: רשימת floats; כל float >= 1

**ברירת מחדל**: `(1, 1.5, 2)` עבור `PEA`, ו-`(1, 3, 5)` אחרת

  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`resilience_level`">

כמות החוסן לבנות נגד שגיאות. רמות גבוהות יותר מייצרות תוצאות מדויקות יותר על חשבון זמני עיבוד ארוכים יותר. ראה את הסעיף [רמות חוסן](/guides/estimator-noise-management#resilience) בנושא ניהול רעש למידע נוסף.

**אפשרויות**: `0`, `1`, `2`

**ברירת מחדל**: `1`

[תיעוד API של `resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
  </AccordionItem>

<AccordionItem title="`seed_estimator`">

**אפשרויות**: מספר שלם

**ברירת מחדל**: None

[`seed_estimator`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#seed_estimator)
  </AccordionItem>

<AccordionItem title="`simulator`">

אפשרויות להעברה בעת סימולציה של Backend

[תיעוד API של `simulator`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-simulator-options)
<Accordion>
<AccordionItem title="`simulator.basis_gates`">

**אפשרויות**: רשימת שמות שערי בסיס להרחבה אליהם

**ברירת מחדל**: קבוצת כל שערי הבסיס הנתמכים על ידי [סימולטור Qiskit Aer](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html)

  </AccordionItem>

<AccordionItem title="`simulator.coupling_map`">

**אפשרויות**: רשימת אינטראקציות של שני קיוביטים מכוונות

**ברירת מחדל**: None, שמשמעותו אין אילוצי קישוריות (קישוריות מלאה).

  </AccordionItem>

<AccordionItem title="`simulator.noise_model`">

**אפשרויות**: [Qiskit Aer NoiseModel](/guides/build-noise-models), או ייצוגו

**ברירת מחדל**: None

  </AccordionItem>

<AccordionItem title="`simulator.seed_simulator`">

**אפשרויות**: מספר שלם

**ברירת מחדל**: None

  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`twirling`">

אפשרויות Twirling

[תיעוד API של `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)
<Accordion>
<AccordionItem title="`twirling.enable_gates`">

**אפשרויות**: True, False

**ברירת מחדל**: False

  </AccordionItem>

<AccordionItem title="`twirling.enable_measure`">

**אפשרויות**: True, False

**ברירת מחדל**: True

  </AccordionItem>

<AccordionItem title="`twirling.num_randomizations`">

**אפשרויות**: `auto`, מספר שלם >= 1

**ברירת מחדל**: `auto`

  </AccordionItem>

<AccordionItem title="`twirling.shots_per_randomization`">

**אפשרויות**: `auto`, מספר שלם >= 1

**ברירת מחדל**: `auto`

  </AccordionItem>

<AccordionItem title="`twirling.strategy`">

**אפשרויות**: `active`, `active-circuit`, `active-accum`, `all`

**ברירת מחדל**: `active-accum`

  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`experimental`">

אפשרויות ניסיוניות, כשזמינות.

  </AccordionItem>

</Accordion>
<span id="options-compatibility-table"></span>
## תאימות תכונות
לא ניתן להשתמש בתכונות Runtime מסוימות יחד במשימה אחת. לחץ על הכרטיסייה המתאימה לקבלת רשימת התכונות שאינן תואמות לתכונה הנבחרת:

<Tabs>

  <TabItem value="Fractional" label="שערים שברייים">
  לא תואם עם:
  - Gate twirling
  - PEA
  - PEC

  </TabItem>

  <TabItem value="ZNE" label="Gate-folding ZNE">
    לא תואם עם:
  - PEA
  - PEC

  עלול לא לעבוד בעת שימוש בשערים מותאמים אישית.
  </TabItem>
  <TabItem value="Twirling" label="Gate twirling">
  לא תואם עם שערים שברייים או עם stretches.

  הערות נוספות:
  - Measurement twirling ניתן להחיל רק על מדידות סופיות.
  - לא עובד עם משלבים שאינם Clifford.

  </TabItem>

  <TabItem value="PEA" label="PEA">
    לא תואם עם:
  - שערים שברייים
  - Gate-folding ZNE
  - PEC
  </TabItem>

  <TabItem value="PEC" label="PEC">
    לא תואם עם:
  - שערים שברייים
  - Gate-folding ZNE
  - PEA
  </TabItem>

</Tabs>

## השלבים הבאים
> **Tip:** - עיין במדריך [מבוא לאפשרויות](/guides/runtime-options-overview).
> - מצא פרטים נוספים על שיטות `EstimatorV2` ב-[מקור ה-API של Estimator](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2).
> - החלט באיזה [מצב ביצוע](/guides/execution-modes) להריץ את המשימה שלך.
> - למד על [ניהול רעש עם Estimator](/guides/estimator-noise-management).

In [3]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator, QiskitRuntimeService

# Define the service.  This allows you to access an IBM QPU.
service = QiskitRuntimeService()

# Get a backend
backend = service.least_busy(operational=True, simulator=False)

# Define Estimator
estimator = Estimator(backend)

options = estimator.options

# Turn off all error mitigation and suppression
options.resilience_level = 0

<span id="options-table"></span>
## Available options

The following table documents options from the latest version of `qiskit-ibm-runtime`. To see older option versions, visit the [`qiskit-ibm-runtime` API reference](/docs/api/qiskit-ibm-runtime) and select a previous version.

<Accordion>
<AccordionItem title="`default_shots`">

The total number of shots to use per circuit per configuration.

**Choices**: Integer >= 0

**Default**: None

[`default_shots` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#default_shots)
  </AccordionItem>


<AccordionItem title="`default_precision`">

The default precision to use for any PUB or `run()` call that does not specify one.

**Choices**: Float > 0

**Default**: 0.015625 (1 / sqrt(4096))

[`default_precision` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#default_precision)
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling`">

Control dynamical decoupling error mitigation settings.

[`dynamical_decoupling` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#dynamical_decoupling)
<Accordion>
<AccordionItem title="`dynamical_decoupling.enable`">

**Choices**: `True`, `False`

**Default**: `False`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.extra_slack_distribution`">

**Choices**: `middle`, `edges`

**Default**: `middle`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.scheduling_method`">

Choices: `asap`, `alap`
Default: `alap`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.sequence_type`">

Choices: `XX`, `XpXm`, `XY4`
Default: `XX`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.skip_reset_qubits`">

Choices: `True`, `False`
Default: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`environment`">

[`environment` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#environment)
<Accordion>
<AccordionItem title="`environment.callback`">

Callable function that receives the `Job ID` and `Job result`.

**Choices**: None

**Default**: None
  </AccordionItem>


<AccordionItem title="`environment.job_tags`">

List of tags.

**Choices**: None

**Default**: None
  </AccordionItem>


<AccordionItem title="`environment.log_level`">

**Choices**: DEBUG, INFO, WARNING, ERROR, CRITICAL

**Default**: WARNING
  </AccordionItem>


<AccordionItem title="`environment.private`">

**Choices**: `True`, `False`

**Default**: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`execution`">

[`execution` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#execution)
<Accordion>
<AccordionItem title="`execution.init_qubits`">
Whether to reset the qubits to the ground state for each shot.

**Choices**: `True`, `False`

**Default**: `True`
  </AccordionItem>


<AccordionItem title="`execution.rep_delay`">
The delay between a measurement and the subsequent quantum circuit.

**Choices**: Value in the range supplied by `backend.rep_delay_range`

**Default**: Given by `backend.default_rep_delay`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`max_execution_time`">
Limits how long a job can run, in seconds. See the [maximum execution time](/docs/guides/max-execution-time) guide for details.

**Choices**: Integer number of seconds in the range [1, 10800]

**Default**: 10800 (3 hours)
  </AccordionItem>


<AccordionItem title="`resilience`">
Advanced resilience options to fine tune the resilience strategy.

[`resilience` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#resilience)
<Accordion>
<AccordionItem title="`resilience.layer_noise_learning`">

Options for learning layer noise.

[`resilience.layer_noise_learning` API documentation](/docs/api/qiskit-ibm-runtime/options-layer-noise-learning-options)
  </AccordionItem>


<AccordionItem title="`resilience.layer_noise_learning.layer_pair_depths`">

**Choices**: list[int] of 2-10 values in the range [0, 200]

**Default**: `(0, 1, 2, 4, 16, 32)`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_learning.max_layers_to_learn`">

**Choices**: None, Integer >= 1

**Default**: `4`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_learning.num_randomizations`">

**Choices**: Integer >= 1

**Default**: `32`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_learning.shots_per_randomization`">

**Choices**: Integer >= 1

**Default**: `128`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_model`">

**Choices**: `NoiseLearnerResult`, `Sequence[LayerError]`

**Default**: None

  </AccordionItem>



<AccordionItem title="`resilience.measure_mitigation`">

**Choices**: `True`, `False`

**Default**: `True`

  </AccordionItem>



<AccordionItem title="`resilience.measure_noise_learning`">

Options for measurement noise learning.

[`resilience.measure_noise_learning` API documentation](/docs/api/qiskit-ibm-runtime/options-measure-noise-learning-options)
  </AccordionItem>


<AccordionItem title="`resilience.measure_noise_learning.num_randomizations`">

**Choices**: Integer >= 1

**Default**: `32`

  </AccordionItem>



<AccordionItem title="`resilience.measure_noise_learning.shots_per_randomization`">

**Choices**: Integer, `auto`

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`resilience.pec_mitigation`">

**Choices**: `True`, `False`

**Default**: `False`

  </AccordionItem>



<AccordionItem title="`resilience.pec`">

Probabilistic error cancellation mitigation options.

[`resilience.pec` API documentation](/docs/api/qiskit-ibm-runtime/options-pec-options)
  </AccordionItem>


<AccordionItem title="`resilience.pec.max_overhead`">

**Choices**: `None`, Integer >= 1


**Default**: `100`

  </AccordionItem>



<AccordionItem title="`resilience.pec.noise_gain`">

**Choices**: `auto`, float in the range [0, 1]

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`resilience.zne_mitigation`">

**Choices**: `True`, `False`

**Default**: `False`

  </AccordionItem>



<AccordionItem title="`resilience.zne`">

[`resilience.zne` API documentation](/docs/api/qiskit-ibm-runtime/options-zne-options)
  </AccordionItem>


<AccordionItem title="`resilience.zne.amplifier`">

**Choices**: `gate_folding`, `gate_folding_front`, `gate_folding_back`, `pea`

**Default**: `gate_folding`

  </AccordionItem>



<AccordionItem title="`resilience.zne.extrapolated_noise_factors`">

**Choices**: List of floats

**Default**: `[0, *noise_factors]`

  </AccordionItem>



<AccordionItem title="`resilience.zne.extrapolator`">

**Choices**: One or more of: `exponential`, `linear`, `double_exponential`, `polynomial_degree_(1 <= k <= 7)`, `fallback`

**Default**: `(exponential, linear)`

  </AccordionItem>



<AccordionItem title="`resilience.zne.noise_factors`">

**Choices**: List of floats; each float >= 1

**Default**: `(1, 1.5, 2)` for `PEA`, and `(1, 3, 5)` otherwise

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`resilience_level`">

How much resilience to build against errors. Higher levels generate more accurate results at the expense of longer processing times. See the [resilience levels](/docs/guides/estimator-noise-management#resilience) section in the Noise management topic to learn more.

**Choices**: `0`, `1`, `2`

**Default**: `1`

[`resilience_level` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
  </AccordionItem>


<AccordionItem title="`seed_estimator`">

**Choices**: Integer

**Default**: None

[`seed_estimator`](/docs/api/qiskit-ibm-runtime/options-estimator-options#seed_estimator)
  </AccordionItem>


<AccordionItem title="`simulator`">

Options to pass when simulating a backend

[`simulator` API documentation](/docs/api/qiskit-ibm-runtime/options-simulator-options)
<Accordion>
<AccordionItem title="`simulator.basis_gates`">

**Choices**: List of basis gate names to unroll to

**Default**: The set of all basis gates supported by [Qiskit Aer simulator](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html)

  </AccordionItem>



<AccordionItem title="`simulator.coupling_map`">

**Choices**: List of directed two-qubit interactions

**Default**: None, which implies no connectivity constraints (full connectivity).

  </AccordionItem>



<AccordionItem title="`simulator.noise_model`">

**Choices**: [Qiskit Aer NoiseModel](/docs/guides/build-noise-models), or its representation

**Default**: None

  </AccordionItem>



<AccordionItem title="`simulator.seed_simulator`">

**Choices**: Integer

**Default**: None

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`twirling`">

Twirling options

[`twirling` API documentation](/docs/api/qiskit-ibm-runtime/options-twirling-options)
<Accordion>
<AccordionItem title="`twirling.enable_gates`">

**Choices**: True, False

**Default**: False

  </AccordionItem>



<AccordionItem title="`twirling.enable_measure`">

**Choices**: True, False

**Default**: True

  </AccordionItem>



<AccordionItem title="`twirling.num_randomizations`">

**Choices**: `auto`, Integer >= 1

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`twirling.shots_per_randomization`">

**Choices**: `auto`, Integer >= 1

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`twirling.strategy`">

**Choices**: `active`, `active-circuit`, `active-accum`, `all`

**Default**: `active-accum`

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`experimental`">

Experimental options, when available.

  </AccordionItem>


</Accordion>

<span id="options-compatibility-table"></span>
## Feature compatibility

Certain runtime features cannot be used together in a single job. Click the appropriate tab for a list of features that are incompatible with the selected feature:

<Accordion>
  <AccordionItem title="Fractional gates">
    Incompatible with:
  - Gate twirling
  - PEA
  - PEC
  </AccordionItem>
  <AccordionItem title="Gate-folding ZNE">
    Might not work when using custom gates. Incompatible with:
  - PEA
  - PEC
  </AccordionItem>
  <AccordionItem title="Gate twirling">
    Incompatible with:
  - Fractional gates
  - Stretches

  Other notes:
  - Measurement twirling can only be applied to terminal measurements.
  - Does not work with non-Clifford entanglers.
  </AccordionItem>
  <AccordionItem title="PEA">
    Incompatible with:
  - Fractional gates
  - Gate-folding ZNE
  - PEC
  </AccordionItem>
  <AccordionItem title="PEC">
    Incompatible with:
  - Fractional gates
  - Gate-folding ZNE
  - PEA
  </AccordionItem>
</Accordion>

## Next steps

<Admonition type="tip" title="Recommendations">
    - Find more details about the `EstimatorV2` methods in the [Estimator API reference](/docs/api/qiskit-ibm-runtime/estimator-v2).
    - Decide what [execution mode](/docs/guides/execution-modes) to run your job in.
    - Learn about [noise management with Estimator](/docs/guides/estimator-noise-management).
</Admonition>